# Session 4 - Regression, Baselines & Honest Error

**Block 2: Machine Learning** · 4 hours

---

## Learning objectives

By the end of this session you will be able to:

1. Derive the ordinary least squares solution, and state what each assumption
   buys you. `[CLO6]`
2. Establish and interpret a baseline before fitting any model. `[CLO1, CLO5]`
3. Compute and interpret MSE, RMSE, MAE and R², and give a scenario in which each
   one misleads. `[CLO5]`
4. Diagnose model misspecification from a residual plot. `[CLO2, CLO6]`
5. Explain overfitting in terms of bias and variance, and produce the curve that
   demonstrates it. `[CLO6]`

## Prerequisites

Sessions 1–3, and your audited pipeline from M2. Also `00c_NumPy_Essentials` - we
will write matrix algebra today.

## Why does this matter?

Linear regression is two hundred years old and it is still the first thing a
competent analyst tries. Not because it wins, but because it is the only model in
this course whose every coefficient you can read, defend, and be held accountable
for.

Today is also where we stop reporting scores and start reporting **errors in units
the client understands**. "R² = 0.81" tells Client B nothing. "Typically within
about −23% to +30% of the right price - so a €177 flat lands somewhere between €137
and €229" tells them whether they can use it. (It is the same model. Only the
sentence changed.)

## §1 - Retrieval practice

From memory. Five minutes.

1. Name the three leaks we removed in Session 3 and the mechanism of each.
2. `price_quote_total_price` correlates about −0.05 with the target. Why was it a leak?
3. Why does scaling transform KNN but not a decision tree?
4. What did our best honest model score at the end of Session 3?
5. What question is still open from Session 3?

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
# DummyRegressor is the baseline: a model that ignores the features entirely.
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import (GroupKFold, cross_val_predict, cross_val_score,
                                     cross_validate)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

from src.data import SEED, load_raw, set_seed, split_by_host

sns.set_theme(style="whitegrid")
set_seed(SEED)

# The whole of Sessions 2 and 3 compressed into one block: split by host, parse the
# price, scope to plausible values. Nothing new happens here, and nothing here is
# allowed to see the sealed test set.
df = load_raw()
train, _ = split_by_host(df)
train["price_num"] = (
    train["price"].astype(str).str.replace(r"[^0-9.]", "", regex=True)
    .replace("", np.nan).astype(float)
)
work = train[train["price_num"].notna()].copy()
work = work[work["price_num"].between(10, 2000)]

y = np.log1p(work["price_num"])
groups = work["host_id"]
gkf = GroupKFold(n_splits=5)

# The audited feature list from Session 3. minimum_nights is in here and doing more
# work than it should. That is deliberate, and Session 6 deals with it.
NUMERIC = ["accommodates", "bedrooms", "beds", "bathrooms", "latitude", "longitude",
           "minimum_nights", "number_of_reviews", "review_scores_rating",
           "calculated_host_listings_count", "hosts_time_as_host_years"]
CATEGORICAL = ["room_type", "property_type", "neighbourhood_cleansed"]

# Same preprocessor as Session 3, reused unchanged so that every score in this
# notebook differs only by the model, never by the preparation.
preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore",
                                               min_frequency=20,
                                               sparse_output=False))]), CATEGORICAL),
])

X = work[NUMERIC + CATEGORICAL]
print(f"modelling rows: {len(work):,}   target: log1p(price)")

## §2 - Baselines first

Before you fit anything, answer this: **what score would a model get if it knew
nothing?**

Without that number, no result you produce afterwards has a meaning. "R² = 0.81"
is not good or bad until you know what zero effort achieves.

In [ ]:
def report(model, label, cols=None):
    """Cross-validated R², RMSE and MAE for one model. Everything in log space."""
    data = X if cols is None else work[cols]
    # Accept either a bare estimator or a ready-made Pipeline, so the same helper
    # can score a plain model and a custom pipeline without special-casing.
    pipe = model if isinstance(model, Pipeline) else Pipeline(
        [("prep", preprocessor), ("model", model)])
    # sklearn maximises everything, so error metrics arrive negated. The "neg_"
    # prefix is the convention, and the minus signs below undo it.
    scoring = ["r2", "neg_root_mean_squared_error", "neg_mean_absolute_error"]
    # cross_validate rather than cross_val_score because we want three metrics from
    # one pass over the folds instead of three separate passes.
    cv = cross_validate(pipe, data, y, cv=gkf, groups=groups, scoring=scoring)
    r2, rmse, mae = (cv["test_r2"], -cv["test_neg_root_mean_squared_error"],
                     -cv["test_neg_mean_absolute_error"])
    # The standard deviation across folds is printed next to R² on purpose. It is
    # the yardstick for deciding whether a later improvement is real.
    print(f"  {label:24s} R² = {r2.mean():+.4f} +/- {r2.std():.4f}    "
          f"RMSE = {rmse.mean():.4f}    MAE = {mae.mean():.4f}")
    return r2.mean(), rmse.mean(), mae.mean()


# The do-nothing model, scored first and on purpose. Until this number exists, no
# later score has a meaning.
print("BASELINE")
# strategy="mean" predicts the training mean for every row, whatever the features.
report(DummyRegressor(strategy="mean"), "predict the mean")

R² of a mean-predictor is **about zero by construction** - R² is *defined* against
that baseline. Ours is very slightly negative (−0.011), and that is informative:
R² is computed per fold against *that fold's* mean, and because we group by host,
the folds have genuinely different average prices. A mean-predictor trained on four
folds is slightly wrong about the fifth.

Keep the baseline's **RMSE of 0.829** in mind. That is the error to beat.

## §3 - Linear regression, derived

### The model

We assume the target is a weighted sum of the features, plus an error:

$$ y_i = \beta_0 + \beta_1 x_{i1} + \dots + \beta_p x_{ip} + \varepsilon_i $$

In matrix form, with a column of ones absorbed into $\mathbf{X}$:

$$ \mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \boldsymbol{\varepsilon} $$

### The cost function

We choose $\boldsymbol{\beta}$ to minimise the sum of squared residuals:

$$ J(\boldsymbol{\beta}) = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2
   = \|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|^2 $$

**Why squared, and not absolute?** Three honest reasons, in decreasing order of
how much they should convince you:

1. It is differentiable everywhere, so the minimum has a closed form. Absolute
   error does not.
2. Under Gaussian errors it is the maximum-likelihood estimator.
3. Historical inertia.

The cost: squaring makes large errors dominate, so OLS is **sensitive to
outliers**. That is a real weakness, not a footnote - and it is why we looked at
the €10,542 listing in Session 2.

### The solution

Expand the cost:

$$ J(\boldsymbol{\beta}) = (\mathbf{y}-\mathbf{X}\boldsymbol{\beta})^{\!\top}
   (\mathbf{y}-\mathbf{X}\boldsymbol{\beta})
   = \mathbf{y}^{\!\top}\mathbf{y}
     - 2\boldsymbol{\beta}^{\!\top}\mathbf{X}^{\!\top}\mathbf{y}
     + \boldsymbol{\beta}^{\!\top}\mathbf{X}^{\!\top}\mathbf{X}\boldsymbol{\beta} $$

Differentiate with respect to $\boldsymbol{\beta}$ and set to zero:

$$ \frac{\partial J}{\partial \boldsymbol{\beta}}
   = -2\mathbf{X}^{\!\top}\mathbf{y} + 2\mathbf{X}^{\!\top}\mathbf{X}\boldsymbol{\beta} = 0 $$

which gives the **normal equation**:

$$ \boxed{\;\hat{\boldsymbol{\beta}} = (\mathbf{X}^{\!\top}\mathbf{X})^{-1}\mathbf{X}^{\!\top}\mathbf{y}\;} $$

This is worth sitting with. Linear regression has an **exact algebraic solution**.
There is no iteration, no learning rate, no convergence to check. It is the only
model in this course with that property, and in Session 8 you will see what we give
up when we lose it.

### Verify it yourself

### What least squares is actually minimising

![estimating coefficients](../assets/diagrams/s04_regression/estimating_coefficients.png)

The vertical segments are the **residuals**: the gap between each observed point and the line's prediction for it. Ordinary least squares chooses the line that makes the sum of their *squares* as small as possible.

Look at the longest segment. Squaring it means that single point contributes far more to the cost than several small errors put together, so the fitted line is pulled towards it. That is the whole mechanism behind the sentence *OLS is sensitive to outliers*, and it is why the EUR 10,542 listing in Session 2 was worth arguing about.

### What the two parameters mean

![slope intercept](../assets/diagrams/s04_regression/slope_intercept.png)

The intercept is the height of the line where x = 0. The slope is the rise over the run: how much the prediction moves for a one-unit change in the feature. Two numbers, and between them everything a straight line is able to say.

Hold on to the slope. In a moment we will fit one on `log1p(price)`, and then the same number will have to be read as a *percentage* change rather than an absolute one.

In [ ]:
# TODO: Compute the OLS coefficients from the normal equation, using only NumPy,
# and confirm they match sklearn's LinearRegression.
#
# Steps:
#   1. Take three numeric features. Handle missing values (median is fine).
#   2. Add a column of ones for the intercept.
#   3. Apply beta = (X'X)^-1 X'y   -- use np.linalg.solve, not np.linalg.inv
#   4. Fit sklearn's LinearRegression on the same data and compare.
#
# Why solve() and not inv()? Answer that in a comment when you find out.

### Reading a coefficient - and being unsettled by it

Because the target is `log1p(price)`, a coefficient $\beta$ means the price is
multiplied by $e^{\beta}$. So:

| feature | coefficient | multiplicative | reads as |
|---|---:|---:|---|
| `accommodates` | **+0.293** | ×1.34 | +34% per additional guest |
| `bedrooms` | **−0.112** | ×0.89 | **−11% per additional bedroom** |
| `bathrooms` | **−0.070** | ×0.93 | **−7% per additional bathroom** |

Read the last two rows again. **More bedrooms predicts a *lower* price. More
bathrooms predicts a *lower* price.**

No sane person believes that adding a bathroom to a Barcelona flat reduces what it
can charge. So either the model is broken, or we are reading it wrong.

We are reading it wrong. Every coefficient in a multiple regression is
**conditional on the others being held fixed**. The `bedrooms` coefficient does not
answer "what happens if I add a bedroom?" It answers:

> *Among listings that sleep the same number of guests, what is associated with
> having one more bedroom?*

And the answer to *that* is: a flat sleeping six people in three bedrooms is a
different kind of property from a flat sleeping six in one - it is more likely to be
a shared or partitioned unit, subdivided for budget travellers. Conditional on
capacity, more bedrooms is a marker of *cheaper* accommodation. The coefficient is
telling the truth about a question we did not think we were asking.

> **This is the most important thing in this section.** A coefficient is not an
> effect. It is an association *conditional on the rest of the model*, and its sign
> can reverse when you add or remove a correlated feature. If your interpretation of
> a coefficient does not name what is being held fixed, it is not an interpretation.

The technical name for what you are seeing is a suppression or
multicollinearity artifact - `accommodates`, `bedrooms` and `bathrooms` all measure
"size" and carry much of the same information (the condition number of
$\mathbf{X}^\top\mathbf{X}$ printed above is a first diagnostic for this).

### Test it

If the negative sign really is an artifact of conditioning on capacity, then
**removing `accommodates` should flip it positive**. That is a falsifiable
prediction. Check it.

In [ ]:
# TODO: Fit log1p(price) on each of these feature sets and print the coefficients:
#
#     ['accommodates', 'bedrooms', 'bathrooms']
#     ['bedrooms', 'bathrooms']
#     ['bedrooms']
#
# Does the sign of `bedrooms` behave as predicted? Write one sentence saying what
# this means for anyone who reads a single coefficient out of a fitted model.

## §4 - Metrics, and what they mean in euros

| Metric | Definition | Units | Reads as |
|---|---|---|---|
| MSE | mean of squared errors | target² | hard to interpret directly |
| RMSE | √MSE | target | typical error, outlier-sensitive |
| MAE | mean absolute error | target | typical error, outlier-resistant |
| R² | 1 − SSE/SST | none | fraction of variance explained vs the mean |

The trap: **our target is not euros, it is log euros.** An RMSE of 0.36 is not "36
cents" and not "36%". It has to be translated.

### R2, as two areas

![Coefficient of Determination](../assets/diagrams/s04_regression/Coefficient_of_Determination.svg)

The larger area is the total variation in the target. The smaller one is the part the model accounts for. R2 is the ratio between them.

This is why a do-nothing baseline scores about zero. R2 is **defined** against predicting the mean, so a mean-predictor sits at zero by construction rather than by coincidence. Ours came out at -0.011, slightly below zero, because each fold is scored against *its own* mean and the folds are grouped by host.

### Predict before you run

Our model will reach RMSE ≈ 0.36 in log space. Client B asks: *"so how wrong is it,
in euros, for a typical €177 listing?"*

Write your answer before running the next cell.

In [ ]:
# TODO: Your prediction, then compute it.
#
# My prediction: for a EUR 177 listing the model is typically wrong by about EUR ___
#
# Hint: if log(y_hat) - log(y) = e, then y_hat / y = exp(e). The error is
# multiplicative, not additive.

**That** is a sentence Client B can act on. "Typically within about €137 to €229
for a €177 flat" tells a pricing manager immediately whether this tool replaces
their judgement or merely informs it. (It informs it.)

Notice the error is **multiplicative**: the model is off by a *percentage*, so it is
wrong by more euros on expensive listings. That is a direct consequence of modelling
the log, and it is usually the right choice for prices - but it must be disclosed.

### Where each metric misleads

| Metric | Fails when | Concrete example here |
|---|---|---|
| R² | comparing across different targets or populations | our R² would change if we re-scoped the target, with no change in model quality |
| RMSE | the target has heavy tails | one €10,542 listing moves it noticeably; MAE barely reacts |
| MAE | you genuinely care about large errors more | mispricing a €2,000 penthouse by 50% matters more than a €40 room |
| all of them | reported without a baseline | RMSE 0.36 means nothing until you know the baseline is 0.83 |

## §5 - Guided: baseline → linear → regularised, with residuals

In [ ]:
# The progression, all five models on identical folds and identical preprocessing.
# Read down the R² column: the jump is from baseline to OLS, and everything after
# that is within the fold-to-fold spread printed beside it.
print("MODEL PROGRESSION")
report(DummyRegressor(strategy="mean"), "1. baseline (mean)")
report(LinearRegression(), "2. OLS")
# Ridge adds a penalty on the size of the coefficients, alpha controlling how hard.
report(Ridge(alpha=1.0), "3. Ridge (α=1)")
# Ten times the penalty, and still essentially the same score: there is no
# overfitting here for regularisation to fix.
report(Ridge(alpha=10.0), "4. Ridge (α=10)")
# Lasso penalises differently and can drive coefficients to exactly zero.
report(Lasso(alpha=0.01), "5. Lasso (α=0.01)")

### Read that table carefully

OLS 0.8078. Ridge(α=1) 0.8080. Ridge(α=10) 0.8082. Lasso 0.7820.

**Regularisation does nothing here.** Three of those numbers are identical to three
decimal places, and the fourth is worse.

You have probably been taught that Ridge and Lasso "fix overfitting". They do - but
there is no overfitting here to fix. We have about 11,000 rows and 78 features
after encoding; a linear model in that regime has plenty of data and is not
straining. Regularisation is a treatment for a disease this patient does not have.

> **Every technique answers a specific problem. Applying it when the problem is
> absent does not help, and Lasso shows it can hurt.**

Lasso is worse because it shrinks coefficients to exactly zero, discarding
genuinely useful weak predictors - with α=0.01 it is deleting information we want.

### Residual diagnostics

The score tells you *how much* you are wrong. The residuals tell you *how* you are
wrong, which is what you can act on.

### Why lasso zeroes coefficients and ridge does not

![L1 vs L2](../assets/diagrams/s04_regression/L1_vs_L2.png)

The red ellipses are contours of the squared error: every point on one ellipse costs the same. The shaded region is the constraint the penalty imposes - a diamond for lasso, a circle for ridge. The fitted coefficients are where the smallest reachable ellipse first touches the region.

A diamond has **corners**, and a corner sits on an axis, which means one coefficient is exactly zero. A circle has no corners, so ridge shrinks coefficients towards zero without ever arriving. That single geometric difference is the whole of *lasso selects features, ridge only shrinks them*.

In [ ]:
# TODO: Produce three residual diagnostics for the Ridge model and interpret each.
#
#   1. residuals vs predicted values  - looking for pattern and for changing spread
#   2. a histogram of residuals        - looking for skew and heavy tails
#   3. residual standard deviation by predicted quintile - quantifying (2)
#
# For each, write what you see AND what it implies about the model.
#
# Use cross_val_predict so that every prediction is out-of-fold.

Three findings, three different implications:

**1. The mean residual is ≈ 0.** Expected - OLS guarantees it on the training data,
and it approximately holds out of fold. If it did not, something would be badly
wrong.

**2. The residuals are right-skewed (+1.04).** This is the interesting one. We took
a log *specifically* to remove skew, and the target's skew did drop from 9.04 to
0.03. But the **residual** skew is still +1.0. Fixing the marginal distribution of
the target did not make the errors symmetric. There remain listings that are far
more expensive than their features can explain - the ones with something we have not
measured. RMSE, which squares, is more affected by these than MAE.

**3. The error spread is largest for the cheapest predictions** (sd 0.43 in the
lowest quintile against 0.30 in Q4). The model is *worse at pricing cheap listings*.
That is a fact about the world you can report: budget rooms are priced less
consistently, presumably because they compete on things this dataset does not
record.

That last one is a genuine finding for Client B, and it came from a residual plot
rather than from a score.

### The OLS assumptions, and which ones we just violated

| Assumption | What it buys | Our status |
|---|---|---|
| Linearity in parameters | the model is identifiable | fine - we chose it |
| Errors have mean zero | coefficients are unbiased | fine (≈ 0.00) |
| **Homoscedasticity** | standard errors are valid | **violated** - see quintile 1 |
| Independent errors | standard errors are valid | **violated** - hosts cluster |
| No perfect collinearity | $\mathbf{X}^\top\mathbf{X}$ is invertible | fine |
| Normal errors | *only* needed for exact small-sample inference | violated (skew +1.04) |

**What the violations cost us.** Not the predictions: OLS remains a perfectly
reasonable predictor with heteroscedastic, non-normal, clustered errors. What they
cost is **inference** - any p-value or confidence interval on a coefficient is
untrustworthy here.

This is worth being precise about, because it is a common muddle. We are doing
prediction. We are allowed to violate the assumptions that only inference needs, so
long as we do not then make inferential claims. The moment you say "bedrooms has a
significant effect", you have crossed a line the data does not support.

## §6 - Bias, variance, and the collapse

Overfitting is usually explained with a picture and a warning. We are going to make
it happen, on purpose, and measure it.

Take four numeric features and expand them into polynomial terms of increasing
degree. Degree 1 is the plain linear model. Degree 5 includes every product of up to
five of those features.

- **High bias / underfitting:** the model is too rigid to capture the pattern. Both
  training and validation scores are poor, and close together.
- **High variance / overfitting:** the model fits the training data's noise.
  Training score keeps rising; validation score falls. The gap opens.

### Underfitting beside overfitting

![bias variance](../assets/diagrams/s04_regression/bias-variance.png)

The same points, fitted twice. On the left a line too rigid to follow the shape of the data. On the right a curve flexible enough to pass through almost every point, including the noise.

Before reading on, decide which of the two you would deploy - and then ask which of the two has the lower error *on the points shown*. The answers are not the same, and the gap between them is what the next experiment measures.

### Predict before you run

Sketch, on paper, the training R² and validation R² as degree goes 1 → 5. Mark where
you think validation performance peaks.

In [ ]:
# TODO: Sketch your prediction, then run the sweep.
#
# My prediction: validation R2 peaks at degree ___ and by degree 5 is about ___

### What just happened

| degree | terms | train R² | validation R² |
|---:|---:|---:|---:|
| 1 | 4 | 0.710 | 0.694 |
| 2 | 14 | 0.756 | 0.730 |
| 3 | 34 | 0.770 | **0.740** ← best |
| 4 | 69 | 0.782 | **0.114** |
| 5 | 125 | 0.789 | **−20.43** |

Training R² **rises monotonically** from 0.710 to 0.789. If you only looked at
training performance, degree 5 is your best model. It is, in fact, catastrophic.

A validation R² of **−20** means the model's squared error is twenty-one times worse
than simply predicting the mean. It has not merely failed to generalise; it is
actively harmful. Nothing in the training score hints at this.

**Two lessons, both worth more than the plot:**

1. **Training performance is not evidence.** It is the one number that is guaranteed
   to improve as you add capacity, which makes it useless as a guide.
2. **Failure can be sudden.** Degree 3 is fine and degree 4 is ruined. There is no
   gentle warning slope. This is why you check, every time, rather than reasoning
   about whether a model "should" be complex enough to overfit.

Notice also that the second panel needed a different y-axis. Whenever one point
destroys your plot's scale, that point is the finding.

## §7 - Common mistakes

| Mistake | Why it is tempting | What to do instead |
|---|---|---|
| Reporting R² with no baseline | it looks like a percentage | our baseline is −0.011 with RMSE 0.829. Quote both. |
| Reporting log-space RMSE to a client | it is what the code printed | translate: exp(RMSE) is a multiplicative factor |
| Reading a coefficient as an effect | the word "effect" is everywhere | it is an association, confounded and scale-dependent |
| Applying Ridge reflexively | it is good practice | here it changes R² by 0.0002. Diagnose before treating. |
| Judging by training score | it is right there | it rose monotonically to a model with R² = −20 |
| `np.linalg.inv` for least squares | it matches the formula | `solve` is faster and better conditioned |
| Claiming significance from OLS output | the software prints p-values | our errors are heteroscedastic and clustered. Prediction is fine; inference is not. |

## §8 - Reflection

1. Client B asks for "an accuracy figure" for the pricing tool. Write the two
   sentences you would actually send, using the numbers from §4.
2. The model is least reliable on the cheapest listings. Name one thing you could
   add to the dataset that might fix this, and say how you would test whether it did.
3. We violated homoscedasticity and normality and carried on. Under what
   circumstance would that have been unacceptable?

## §9 - Knowledge check

1. Write the normal equation and say why `np.linalg.solve` is preferred to
   inverting.
2. Our baseline scored R² = −0.011. Why is it not exactly zero?
3. RMSE = 0.36 on a `log1p` target. What does that mean for a €200 listing?
4. Training R² rose from 0.710 to 0.789 while validation fell to −20.4. Explain in
   terms of bias and variance.
5. Name one OLS assumption we violated and state precisely what it did and did not
   invalidate.

## Summary

- **Baseline first.** Mean-prediction gives R² ≈ 0 and RMSE 0.829. Every later
  number is meaningful only against those.
- Linear regression has a **closed-form solution**, the normal equation. It is the
  only model in this course that does, and Session 8 will show what iteration costs.
- A coefficient on a log target is a **percentage** effect - and an association, not
  a cause.
- **Translate your metrics.** RMSE 0.36 in log space is roughly +43%/−30%: a €177
  listing predicted between about €137 and €229. That is the sentence the client
  needs.
- **Regularisation changed nothing** (0.8078 → 0.8082) because there was no
  overfitting to treat, and Lasso made things worse. Diagnose before you treat.
- Residuals gave us a finding no score could: the model is **least reliable on the
  cheapest listings**.
- We violated homoscedasticity, normality and independence. That invalidates
  **inference**, not prediction - provided we do not then claim significance.
- The polynomial sweep took validation R² from 0.740 to **−20.43** while training R²
  kept rising. Training performance is not evidence.

## Key takeaways

1. No score means anything without a baseline.
2. Report error in the client's units, not the model's.
3. Every technique treats a specific problem. Check you have the problem.

## Further exploration

**Essential**
- James et al., *An Introduction to Statistical Learning*, ch. 3 (linear regression)
  and §2.2 (bias–variance). https://www.statlearning.com/
- scikit-learn user guide, *Linear Models*:
  https://scikit-learn.org/stable/modules/linear_model.html

**Recommended**
- Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning*, §3.2 and
  §3.4 for Ridge/Lasso as constrained optimisation.
  https://hastie.su.domains/ElemStatLearn/
- Shmueli (2010), *To Explain or to Predict?*, Statistical Science 25(3) - the
  prediction-versus-inference distinction from §5, argued properly.

**Advanced**
- Belkin et al. (2019), *Reconciling modern machine-learning practice and the
  classical bias–variance trade-off*, PNAS. https://arxiv.org/abs/1812.11118 -
  the classical U-curve you drew in §6 is not the whole story for very
  over-parameterised models. Read it as an open research thread, not a correction.

---

**Next session:** classification, and the discovery that a model doing nothing at
all can score an F1 of 0.80.